In [ ]:
using CUDA, Random, LinearAlgebra

# Vérifier les GPUs disponibles
devices = collect(CUDA.devices())
@assert length(devices) >= 2 "Il faut au moins 2 GPUs"

println("✅ GPUs détectés :")
for (i, dev) in enumerate(devices)
    println("   GPU $i : $(CUDA.name(dev))")
end

# Utilitaires pour changer de GPU
gpu1_idx = 0  # premier GPU
gpu2_idx = 1  # second GPU

function set_gpu(gpu_idx)
    CUDA.device!(gpu_idx)
    println("🔧 GPU actif : $(CUDA.name(CUDA.device()))")
end

# Test rapide : allouer un petit tenseur sur chaque GPU
set_gpu(gpu1_idx)
a = CUDA.rand(Float32, 2,2)
set_gpu(gpu2_idx)
b = CUDA.rand(Float32, 2,2)
println("✅ Test d'allocation réussi sur les deux GPUs.")

In [ ]:
# Cellule 2 : Structure du modèle et initialisation
mutable struct ParallelModel
    W1::CuArray{Float32,2}
    W2::CuArray{Float32,2}
    m1_W1::CuArray{Float32,2}
    m2_W1::CuArray{Float32,2}
    m1_W2::CuArray{Float32,2}
    m2_W2::CuArray{Float32,2}
end

function init_model_on_gpu(gpu_idx::Int, d_in::Int, d_hidden::Int, d_out::Int)
    set_gpu(gpu_idx)
    # He initialization
    W1 = CUDA.randn(Float32, d_hidden, d_in) .* sqrt(2.0 / d_in)
    W2 = CUDA.randn(Float32, d_out, d_hidden) .* sqrt(2.0 / d_hidden)
    # Moments pour AdamW
    m1_W1 = CUDA.zeros(Float32, d_hidden, d_in)
    m2_W1 = CUDA.zeros(Float32, d_hidden, d_in)
    m1_W2 = CUDA.zeros(Float32, d_out, d_hidden)
    m2_W2 = CUDA.zeros(Float32, d_out, d_hidden)
    return ParallelModel(W1, W2, m1_W1, m2_W1, m1_W2, m2_W2)
end

# Initialisation des deux modèles (poids identiques pour démarrer)
model1 = init_model_on_gpu(gpu1_idx, 784, 128, 10)
model2 = init_model_on_gpu(gpu2_idx, 784, 128, 10)
println("✅ Modèles initialisés sur GPU1 et GPU2")

In [ ]:
using MLDatasets

function load_mnist_batches(batch_size::Int)
    # Chargement des données d'entraînement
    train_x, train_y = MNIST.traindata(Float32)
    # train_x: (28,28,60000) -> aplatir en (60000, 784)
    X = reshape(train_x, 784, 60000)'  # (60000, 784)
    Y = train_y  # (60000,) labels 0-9
    
    # Préparation des batches
    n_samples = size(X, 1)
    n_batches = div(n_samples, batch_size)
    
    batches = []
    for i in 1:n_batches
        idx_start = (i-1)*batch_size + 1
        idx_end = i*batch_size
        X_batch = X[idx_start:idx_end, :]
        Y_batch = Y[idx_start:idx_end]
        push!(batches, (X_batch, Y_batch))
    end
    
    return batches
end

function split_batch_for_gpus(x_cpu::Matrix{Float32}, y_cpu::Vector{Int}, batch_size::Int)
    # Divise un batch en deux sous-batchs de taille égale (ou à peu près)
    half = div(batch_size, 2)
    x1 = x_cpu[1:half, :]
    y1 = y_cpu[1:half]
    x2 = x_cpu[half+1:end, :]
    y2 = y_cpu[half+1:end]
    return (x1, y1), (x2, y2)
end

# Test avec batch_size=128
batch_size = 128
batches = load_mnist_batches(batch_size)
println("✅ Chargé $(length(batches)) batches de taille $batch_size")

# Tester le split sur le premier batch
x_cpu, y_cpu = batches[1]
(x1, y1), (x2, y2) = split_batch_for_gpus(x_cpu, y_cpu, batch_size)
println("Batch 1 splitté: sous-batch1 taille $(size(x1,1)), sous-batch2 taille $(size(x2,1))")

In [ ]:
using MLDatasets: MNIST

function load_mnist_batches(batch_size::Int, gpu1_idx::Int, gpu2_idx::Int)
    # Chargement avec API moderne (pas de warning)
    train_x, train_y = MNIST(split=:train)[:]
    # train_x est (28,28,60000) -> (60000, 784)
    train_x = Float32.(reshape(train_x, 28*28, :))'
    train_x = train_x ./ 255.0f0
    # One-hot encoding (60000, 10)
    train_y_onehot = zeros(Float32, size(train_x,1), 10)
    for i in 1:length(train_y)
        train_y_onehot[i, train_y[i]+1] = 1.0f0
    end
    # Shuffle et création des batches
    n = size(train_x, 1)
    indices = shuffle(1:n)
    x_batches = [train_x[indices[i:min(i+batch_size-1, n)], :] for i in 1:batch_size:n]
    y_batches = [train_y_onehot[indices[i:min(i+batch_size-1, n)], :] for i in 1:batch_size:n]
    
    # Fonction pour diviser un batch en deux (un sur chaque GPU)
    function split_batch(x, y)
        n_batch = size(x, 1)
        n_half = div(n_batch, 2)
        # Premier sous-batch (GPU1)
        x1 = CuArray(x[1:n_half, :])
        y1 = CuArray(y[1:n_half, :])
        # Second sous-batch (GPU2)
        set_gpu(gpu2_idx)
        x2 = CuArray(x[n_half+1:end, :])
        y2 = CuArray(y[n_half+1:end, :])
        set_gpu(gpu1_idx)  # revenir au GPU1 pour le reste
        return (x1, y1), (x2, y2)
    end
    
    return x_batches, y_batches, split_batch
end

batch_size = 256
x_batches, y_batches, split_batch = load_mnist_batches(batch_size, gpu1_idx, gpu2_idx)
println("✅ MNIST chargé : $(length(x_batches)) batches de taille $batch_size")

In [ ]:
# Cellule 4 : Fonctions forward/backward pour un GPU
function forward_gpu!(model::ParallelModel, X::CuArray{Float32,2})
    # X shape: (batch_size, 784)
    # W1: (128, 784)
    h = X * model.W1'  # (batch, 128)
    h = max.(0.0f0, h)  # ReLU
    logits = h * model.W2'  # (batch, 10)
    return logits, h
end

function backward_gpu!(model::ParallelModel, X::CuArray{Float32,2}, Y::CuArray{Float32,2}, logits::CuArray{Float32,2}, h::CuArray{Float32,2})
    batch_size = size(X, 1)
    
    # Softmax + cross-entropy gradient
    max_logits = maximum(logits, dims=2)
    exp_logits = exp.(logits .- max_logits)
    probs = exp_logits ./ sum(exp_logits, dims=2)
    d_logits = (probs - Y) ./ batch_size
    
    # Gradient pour W2
    dW2 = d_logits' * h
    # Gradient pour h
    d_h = d_logits * model.W2
    # ReLU derivative using broadcasting but with conditional
    d_h .= d_h .* (h .> 0.0f0)   # remplace d_h[h .<= 0.0f0] = 0.0f0
    
    # Gradient pour W1
    dW1 = d_h' * X
    
    return dW1, dW2
end

function compute_loss(probs::CuArray{Float32,2}, Y::CuArray{Float32,2})
    batch_size = size(probs, 1)
    loss = -sum(Y .* log.(probs .+ Float32(1e-8))) / batch_size
    return loss
end

println("✅ Fonctions forward/backward définies")

In [ ]:
# Cellule 5 : Entraînement parallèle sur deux GPUs
using Statistics

function train_parallel!(model1, model2, x_batches_cpu, y_batches_cpu, split_batch, lr, epochs)
    losses = Float32[]
    for epoch in 1:epochs
        epoch_loss = 0.0f0
        for i in 1:length(x_batches_cpu)
            x_cpu = x_batches_cpu[i]
            y_cpu = y_batches_cpu[i]
            # split_batch renvoie les données CPU (pas encore GPU)
            (x1_cpu, y1_cpu), (x2_cpu, y2_cpu) = split_batch(x_cpu, y_cpu)
            
            # === GPU 1 ===
            CUDA.device!(gpu1_idx)
            x1 = CuArray(x1_cpu)
            y1 = CuArray(y1_cpu)
            logits1, h1 = forward_gpu!(model1, x1)
            loss1 = compute_loss(softmax(logits1), y1)
            dW1_1, dW2_1 = backward_gpu!(model1, x1, y1, logits1, h1)
            
            # === GPU 2 ===
            CUDA.device!(gpu2_idx)
            x2 = CuArray(x2_cpu)
            y2 = CuArray(y2_cpu)
            logits2, h2 = forward_gpu!(model2, x2)
            loss2 = compute_loss(softmax(logits2), y2)
            dW1_2, dW2_2 = backward_gpu!(model2, x2, y2, logits2, h2)
            
            # Transfert des gradients du GPU2 vers GPU1 via CPU
            CUDA.device!(gpu1_idx)  # on revient sur GPU1 pour la suite
            dW1_2_cpu = Array(dW1_2)
            dW2_2_cpu = Array(dW2_2)
            dW1_2_gpu1 = CuArray(dW1_2_cpu)
            dW2_2_gpu1 = CuArray(dW2_2_cpu)
            
            dW1_avg = (dW1_1 .+ dW1_2_gpu1) ./ 2.0f0
            dW2_avg = (dW2_1 .+ dW2_2_gpu1) ./ 2.0f0
            
            # Mise à jour du modèle 1 (sur GPU1)
            CUDA.device!(gpu1_idx)
            update_adamw!(model1, dW1_avg, dW2_avg, lr, epoch * length(x_batches_cpu) + i)
            
            # Mise à jour du modèle 2 (sur GPU2)
            dW1_avg_cpu = Array(dW1_avg)
            dW2_avg_cpu = Array(dW2_avg)
            CUDA.device!(gpu2_idx)
            dW1_avg_gpu2 = CuArray(dW1_avg_cpu)
            dW2_avg_gpu2 = CuArray(dW2_avg_cpu)
            update_adamw!(model2, dW1_avg_gpu2, dW2_avg_gpu2, lr, epoch * length(x_batches_cpu) + i)
            
            epoch_loss += (loss1 + loss2) / 2.0f0
        end
        avg_loss = epoch_loss / length(x_batches_cpu)
        push!(losses, avg_loss)
        println("Epoch $epoch, Loss: $avg_loss")
    end
    return losses
end

println("✅ Fonction d'entraînement parallèle définie")


In [ ]:
function test_single_gpu(model, x_cpu, y_cpu, gpu_idx)
    CUDA.device!(gpu_idx)
    x = CuArray(x_cpu)
    y = CuArray(y_cpu)
    logits, h = forward_gpu!(model, x)
    loss = compute_loss(softmax(logits), y)
    dW1, dW2 = backward_gpu!(model, x, y, logits, h)
    return loss, dW1, dW2
end

batch_idx = 1
x = x_batches[batch_idx]
y = y_batches[batch_idx]
n_half = div(size(x,1), 2)
x1 = x[1:n_half, :]
y1 = y[1:n_half, :]
x2 = x[n_half+1:end, :]
y2 = y[n_half+1:end, :]

println("Test GPU1...")
loss1, _, _ = test_single_gpu(model1, x1, y1, gpu1_idx)
println("Loss GPU1: $loss1")

println("Test GPU2...")
loss2, _, _ = test_single_gpu(model2, x2, y2, gpu2_idx)
println("Loss GPU2: $loss2")

In [ ]:
CUDA.device!(gpu1_idx)
println("GPU1, model1.W1 device: ", CUDA.device(model1.W1))
CUDA.device!(gpu2_idx)
println("GPU2, model2.W1 device: ", CUDA.device(model2.W1))

In [ ]:
using Statistics

function parallel_train_step!(model1::ParallelModel, model2::ParallelModel, 
                              x_batch::Matrix{Float32}, y_batch::Matrix{Float32},
                              split_batch_fn, lr::Float32, t::Int)
    # Diviser le batch en deux
    (x1, y1), (x2, y2) = split_batch_fn(x_batch, y_batch)
    
    # Forward sur GPU1
    set_gpu(gpu1_idx)
    logits1, h1 = forward_gpu!(model1, x1)
    dW1_1, dW2_1 = backward_gpu!(model1, x1, y1, logits1, h1)
    
    # Forward sur GPU2
    set_gpu(gpu2_idx)
    logits2, h2 = forward_gpu!(model2, x2)
    dW1_2, dW2_2 = backward_gpu!(model2, x2, y2, logits2, h2)
    
    # Moyenner les gradients (les ramener sur CPU pour calcul)
    dW1_avg = (Array(dW1_1) + Array(dW1_2)) / 2
    dW2_avg = (Array(dW2_1) + Array(dW2_2)) / 2
    
    # Mise à jour des deux modèles avec les gradients moyennés
    # GPU1
    set_gpu(gpu1_idx)
    adamw_step!(Backend.CPUDevice(), model1.W1, CuArray(dW1_avg), 
                model1.m1_W1, model1.m2_W1, lr, 0.9f0, 0.999f0, 1f-8, Int32(t), 1.0f0, 0.01f0)
    adamw_step!(Backend.CPUDevice(), model1.W2, CuArray(dW2_avg), 
                model1.m1_W2, model1.m2_W2, lr, 0.9f0, 0.999f0, 1f-8, Int32(t), 1.0f0, 0.01f0)
    
    # GPU2
    set_gpu(gpu2_idx)
    adamw_step!(Backend.CPUDevice(), model2.W1, CuArray(dW1_avg), 
                model2.m1_W1, model2.m2_W1, lr, 0.9f0, 0.999f0, 1f-8, Int32(t), 1.0f0, 0.01f0)
    adamw_step!(Backend.CPUDevice(), model2.W2, CuArray(dW2_avg), 
                model2.m1_W2, model2.m2_W2, lr, 0.9f0, 0.999f0, 1f-8, Int32(t), 1.0f0, 0.01f0)
    
    # Retourner la perte moyenne (optionnel)
    set_gpu(gpu1_idx)
    loss1 = compute_loss(softmax(logits1), y1)
    set_gpu(gpu2_idx)
    loss2 = compute_loss(softmax(logits2), y2)
    set_gpu(gpu1_idx)
    return (loss1 + loss2) / 2
end

In [ ]:
function train_parallel!(model1, model2, x_batches_cpu, y_batches_cpu, split_batch, lr, epochs)
    losses = Float32[]
    for epoch in 1:epochs
        epoch_loss = 0.0f0
        for i in 1:length(x_batches_cpu)
            x_cpu = x_batches_cpu[i]
            y_cpu = y_batches_cpu[i]
            # split_batch renvoie les données CPU (pas encore GPU)
            (x1_cpu, y1_cpu), (x2_cpu, y2_cpu) = split_batch(x_cpu, y_cpu)
            
            # === GPU 1 ===
            CUDA.device!(gpu1_idx)
            x1 = CuArray(x1_cpu)
            y1 = CuArray(y1_cpu)
            logits1, h1 = forward_gpu!(model1, x1)
            loss1 = compute_loss(softmax(logits1), y1)
            dW1_1, dW2_1 = backward_gpu!(model1, x1, y1, logits1, h1)
            
            # === GPU 2 ===
            CUDA.device!(gpu2_idx)
            x2 = CuArray(x2_cpu)
            y2 = CuArray(y2_cpu)
            logits2, h2 = forward_gpu!(model2, x2)
            loss2 = compute_loss(softmax(logits2), y2)
            dW1_2, dW2_2 = backward_gpu!(model2, x2, y2, logits2, h2)
            
            # Transfert des gradients du GPU2 vers GPU1 via CPU
            CUDA.device!(gpu1_idx)  # on revient sur GPU1 pour la suite
            dW1_2_cpu = Array(dW1_2)
            dW2_2_cpu = Array(dW2_2)
            dW1_2_gpu1 = CuArray(dW1_2_cpu)
            dW2_2_gpu1 = CuArray(dW2_2_cpu)
            
            dW1_avg = (dW1_1 .+ dW1_2_gpu1) ./ 2.0f0
            dW2_avg = (dW2_1 .+ dW2_2_gpu1) ./ 2.0f0
            
            # Mise à jour du modèle 1 (sur GPU1)
            CUDA.device!(gpu1_idx)
            update_adamw!(model1, dW1_avg, dW2_avg, lr, epoch * length(x_batches_cpu) + i)
            
            # Mise à jour du modèle 2 (sur GPU2)
            dW1_avg_cpu = Array(dW1_avg)
            dW2_avg_cpu = Array(dW2_avg)
            CUDA.device!(gpu2_idx)
            dW1_avg_gpu2 = CuArray(dW1_avg_cpu)
            dW2_avg_gpu2 = CuArray(dW2_avg_cpu)
            update_adamw!(model2, dW1_avg_gpu2, dW2_avg_gpu2, lr, epoch * length(x_batches_cpu) + i)
            
            epoch_loss += (loss1 + loss2) / 2.0f0
        end
        avg_loss = epoch_loss / length(x_batches_cpu)
        push!(losses, avg_loss)
        println("Epoch $epoch, Loss: $avg_loss")
    end
    return losses
end

# Helper pour softmax
function softmax(x::CuArray{Float32,2})
    max_x = maximum(x, dims=2)
    exp_x = exp.(x .- max_x)
    return exp_x ./ sum(exp_x, dims=2)
end

# Mise à jour AdamW (à adapter selon votre fonction existante)
function update_adamw!(model, dW1, dW2, lr, t)
    β1 = 0.9f0
    β2 = 0.999f0
    ε = 1e-8f0
    wd = 0.01f0
    # Mise à jour W1
    model.m1_W1 .= β1 .* model.m1_W1 .+ (1f0 - β1) .* dW1
    model.m2_W1 .= β2 .* model.m2_W1 .+ (1f0 - β2) .* (dW1 .^ 2)
    m1_hat = model.m1_W1 ./ (1f0 - β1^t)
    m2_hat = model.m2_W1 ./ (1f0 - β2^t)
    model.W1 .= model.W1 .* (1f0 - lr * wd) .- lr .* m1_hat ./ (sqrt.(m2_hat) .+ ε)
    # Mise à jour W2
    model.m1_W2 .= β1 .* model.m1_W2 .+ (1f0 - β1) .* dW2
    model.m2_W2 .= β2 .* model.m2_W2 .+ (1f0 - β2) .* (dW2 .^ 2)
    m1_hat = model.m1_W2 ./ (1f0 - β1^t)
    m2_hat = model.m2_W2 ./ (1f0 - β2^t)
    model.W2 .= model.W2 .* (1f0 - lr * wd) .- lr .* m1_hat ./ (sqrt.(m2_hat) .+ ε)
    return nothing
end

# Test sur 5 batches
println("🚀 Test d'entraînement parallèle sur 2 GPUs...")
losses = train_parallel!(model1, model2, x_batches[1:5], y_batches[1:5], split_batch, 0.001f0, 2)
println("✅ Test terminé")